# Quy – Step 6, 6.5, 7, 8: Tìm kiếm bài hát (TF-IDF vs BM25) và so sánh tiền xử lý
**Người làm:** Lữ Phú Quý
Dữ liệu: bảng `songs` trong `music_seg_merged_preprocesing.db` (KHÔNG push file .db).
Văn bản để search = `title + artist + lyrics`. "category" = website nguồn lấy từ url.

Cần để cùng thư mục: `Thanh_step1-2.ipynb`, `Dang_preprocess_tokens.py`, file .db.

In [ ]:
# Colab: !pip install nltk rank_bm25
import sys; sys.path.insert(0, '.')
import os, re, json, html, sqlite3, unicodedata, glob
import numpy as np, pandas as pd
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import nltk   # Dang_preprocess_tokens.py can nltk

try:
    from rank_bm25 import BM25Okapi
    print("Dung rank_bm25.BM25Okapi")
except ImportError:
    # Cai lai dung cong thuc cua rank_bm25.BM25Okapi (idf am -> epsilon * idf trung binh)
    class BM25Okapi:
        def __init__(self, corpus, k1=1.5, b=0.75, epsilon=0.25):
            self.k1, self.b = k1, b
            self.doc_len = np.array([len(d) for d in corpus])
            self.avgdl = self.doc_len.mean()
            self.tf = [Counter(d) for d in corpus]
            N = len(corpus); df = Counter(w for d in corpus for w in set(d))
            idf = {w: np.log(N - n + 0.5) - np.log(n + 0.5) for w, n in df.items()}
            avg_idf = sum(idf.values()) / len(idf)
            self.idf = {w: (v if v >= 0 else epsilon * avg_idf) for w, v in idf.items()}
        def get_scores(self, query):
            s = np.zeros(len(self.tf))
            norm = self.k1 * (1 - self.b + self.b * self.doc_len / self.avgdl)
            for q in query:
                f = np.array([d.get(q, 0) for d in self.tf])
                s += self.idf.get(q, 0) * f * (self.k1 + 1) / (f + norm)
            return s
    print("Khong co rank_bm25 -> dung BM25Okapi tu cai (cung cong thuc)")
print("nltk", nltk.__version__)
if 'SHIM' in str(nltk.__version__): print("CANH BAO: dang dung nltk gia lap -> stemmed/lemmatized chua dung, hay chay lai voi nltk that")

Khong co rank_bm25 -> dung BM25Okapi tu cai (cung cong thuc)
nltk SHIM(khong phai nltk that)
CANH BAO: dang dung nltk gia lap -> stemmed/lemmatized chua dung, hay chay lai voi nltk that


In [ ]:
DB = (glob.glob('*.db') + glob.glob('/mnt/user-data/attachments/*/*.db') + glob.glob('/content/*.db'))[0]
con = sqlite3.connect(DB)
df = pd.read_sql("SELECT id, url, title, artist, lyrics FROM songs", con)
df = df.fillna({'title': '', 'artist': ''})
print(df.shape)

(2765, 5)


## Dùng code của Thành và Đăng (để cùng thư mục với notebook này)
- `clean_text(text, site)` + bộ `QUERIES` (10 truy vấn, nhãn theo id) đọc trực tiếp từ `Thanh_step1-2.ipynb` — chỉ exec đúng 2 ô định nghĩa, không sửa file của Thành.
- `preprocess_tokens(text, strategy)` import từ `Dang_preprocess_tokens.py`.
- Văn bản search = clean_text(title) + clean_text(artist) + clean_text(lyrics, site). Corpus = các bài còn lời sau làm sạch (giống `corpus` của Thành).

In [ ]:
from urllib.parse import urlparse
nb_thanh = json.load(open('Thanh_step1-2.ipynb', encoding='utf-8'))
src = {c: ''.join(cell['source']) for c, cell in enumerate(nb_thanh['cells']) if cell['cell_type'] == 'code'}
ns = {'re': re, 'html': html, 'unicodedata': unicodedata, 'urlparse': urlparse}
exec(next(s for s in src.values() if 'def clean_text' in s).split('# smoke test')[0], ns)
exec('from collections import Counter\n' + next(s for s in src.values() if s.lstrip().startswith('QUERIES = [')), ns)
clean_text, QUERIES = ns['clean_text'], ns['QUERIES']
print("clean_text cua Thanh:", clean_text("Verse 1:\nSơn Tùng M-TP &amp; don&#039;t", "open.spotify.com"))
print(len(QUERIES), "truy van cua Thanh")

from Dang_preprocess_tokens import preprocess_tokens
print("preprocess_tokens cua Dang:", preprocess_tokens(clean_text("Lạc Trôi - Sơn Tùng M-TP, don't cry"), "no_diacritics"))

10 truy vấn; Counter({'title': 5, 'lyric': 3, 'no_accent': 2})
clean_text cua Thanh: sơn tùng m-tp & don't
10 truy van cua Thanh
preprocess_tokens cua Dang: ['lac', 'troi', 'son', 'tung', 'm', 'tp', 'dont', 'cry']


In [ ]:
df['site'] = df['url'].map(lambda u: urlparse(u).netloc.replace('www.', ''))
df['clean_lyrics'] = [clean_text(t, s) for t, s in zip(df['lyrics'], df['site'])]
corpus = df[df['clean_lyrics'].str.len() > 0].reset_index(drop=True)
corpus['search_text'] = [f"{clean_text(t)} {clean_text(a)} {l}" for t, a, l in
                         zip(corpus['title'], corpus['artist'], corpus['clean_lyrics'])]
print("Corpus:", corpus.shape, corpus['site'].value_counts().to_dict())

def tokenize_for_search(text, strategy='baseline', already_clean=False):
    # Dung CHUNG cho bai hat va query: query cung qua clean_text roi preprocess_tokens
    return preprocess_tokens(text if already_clean else clean_text(text), strategy)

print(tokenize_for_search(corpus.loc[0, 'search_text'], already_clean=True)[:25])

Corpus: (2130, 8) {'open.spotify.com': 1566, 'nhaccuatui.com': 315, 'hopamchuan.com': 227, 'nhac.vn': 22}
['sớm', 'muộn', 'thì', 'anh', 'trai', 'say', 'hi', 'hustlang', 'robber', 'nhâm', 'phương', 'nam', 'mason', 'nguyen', 'jaysonlei', 'khoi', 'vu', 'lamoon', 'baby', 'à', 'anh', 'biết', 'là', 'sớm', 'muộn']


## Step 6 – TF-IDF + cosine và BM25 (k1=1.5, b=0.75)

In [ ]:
class SearchEngine:
    def __init__(self, docs_tokens):
        self.docs_tokens = docs_tokens
        self.tfidf = TfidfVectorizer(analyzer=lambda x: x)          # dua vao list token da xu ly san
        self.X = self.tfidf.fit_transform(docs_tokens)
        self.bm25 = BM25Okapi(docs_tokens, k1=1.5, b=0.75)
    def scores(self, q_tokens, method):
        if method == 'tfidf':
            return cosine_similarity(self.tfidf.transform([q_tokens]), self.X).ravel()
        return np.asarray(self.bm25.get_scores(q_tokens))
    def search(self, q_tokens, method, k=5):
        s = self.scores(q_tokens, method)
        idx = np.argsort(-s, kind='stable')[:k]
        return [(int(corpus.loc[i, 'id']), float(s[i])) for i in idx if s[i] > 0]

docs_base = [tokenize_for_search(t, already_clean=True) for t in corpus['search_text']]
engine = SearchEngine(docs_base)
id2row = corpus.set_index('id')

def show(query, method, eng=engine, prep=tokenize_for_search):
    res = eng.search(prep(query), method)
    print(f"\n[{method.upper()}] '{query}'  -> tokens {prep(query)}")
    if not res: print("  (khong co ket qua: moi token deu nam ngoai vocabulary / diem = 0)")
    for r, (i, s) in enumerate(res, 1):
        print(f"  {r}. id={i:<5} {s:7.3f}  {id2row.loc[i,'title'][:45]:<45} | {id2row.loc[i,'artist'][:30]}")

for q in ['Lạc Trôi', 'lac troi', 'nếu ngày mai em rời xa anh', 'em la ai tu dau buoc den noi day', 'xyzqw blorpt']:
    for m in ['tfidf', 'bm25']: show(q, m)


[TFIDF] 'Lạc Trôi'  -> tokens ['lạc', 'trôi']
  1. id=1608    0.460  Lạc Trôi - Sky Tour 2019                      | Sơn Tùng M-TP
  2. id=2729    0.423  Lạc Trôi                                      | Sơn Tùng M-TP
  3. id=403     0.384  Lạc                                           | Lê Hà Nguyên
  4. id=2108    0.355  Lạc Mất Em                                    | Đàm Vĩnh Hưng
  5. id=2708    0.322  Lạc Trôi                                      | Sơn Tùng M-TP

[BM25] 'Lạc Trôi'  -> tokens ['lạc', 'trôi']
  1. id=536     7.076  Lạc Trôi                                      | Sơn Tùng M-TP
  2. id=1128    7.076  Lạc Trôi                                      | Sơn Tùng M-TP
  3. id=1145    7.076  Lạc Trôi                                      | Sơn Tùng M-TP
  4. id=1278    7.076  Lạc Trôi                                      | Sơn Tùng M-TP
  5. id=1608    7.044  Lạc Trôi - Sky Tour 2019                      | Sơn Tùng M-TP

[TFIDF] 'lac troi'  -> tokens ['lac', 'troi']
  (khong co

**Nhận xét Step 6**
- Query toàn từ lạ (`xyzqw blorpt`): vector TF-IDF toàn 0 → cosine = 0 với mọi bài; BM25 cộng idf của từ không có = 0 → không có kết quả. Hệ thống không báo lỗi nhưng trả về rỗng, cần fallback (gợi ý chính tả, bỏ dấu, n-gram ký tự).
- Query gõ không dấu `lac troi` gần như không tìm ra "Lạc Trôi" vì token `lac`, `troi` khác hẳn `lạc`, `trôi` → lý do cần chiến lược `no_diacritics` ở Step 7.

## Step 6.5 – Precision@k, Recall@k (so theo id trong DB)

In [ ]:
QRELS = {q['query']: q['relevant_ids'] for q in QUERIES}   # nhan cua Thanh (id trong DB)
QTYPE = {q['query']: q['type'] for q in QUERIES}
print(len(QRELS), "truy van:", Counter(QTYPE.values()))

def precision_at_k(ret, rel, k=5): return len(set(ret[:k]) & set(rel)) / k
def recall_at_k(ret, rel, k=5):    return len(set(ret[:k]) & set(rel)) / len(rel) if rel else 0.0

def evaluate(eng, prep, method, k=5):
    rows = []
    for q, rel in QRELS.items():
        ret = [i for i, _ in eng.search(prep(q), method, k)]
        rows.append({'query': q, 'type': QTYPE[q], 'P@5': precision_at_k(ret, rel, k), 'R@5': recall_at_k(ret, rel, k), 'top5': ret})
    return pd.DataFrame(rows)

ev_t, ev_b = evaluate(engine, tokenize_for_search, 'tfidf'), evaluate(engine, tokenize_for_search, 'bm25')
cmp_m = ev_t[['query','type','P@5','R@5']].merge(ev_b[['query','P@5','R@5']], on='query', suffixes=('_tfidf','_bm25'))
print(cmp_m.round(3).to_string(index=False))
print("\nTheo loai truy van (BM25 P@5):", cmp_m.groupby('type')['P@5_bm25'].mean().round(3).to_dict())
print("\nTrung binh:\n", cmp_m.drop(columns=['query','type']).mean().round(3).to_string())

10 truy van: Counter({'title': 5, 'lyric': 3, 'no_accent': 2})
                                   query      type  P@5_tfidf  R@5_tfidf  P@5_bm25  R@5_bm25
                                Lạc Trôi     title        0.6      0.375       1.0     0.625
            Chúng Ta Không Thuộc Về Nhau     title        1.0      0.833       1.0     0.833
                      Thiệp Hồng Sai Tên     title        1.0      0.625       1.0     0.625
              Có Chàng Trai Viết Lên Cây     title        1.0      0.714       1.0     0.714
                      Nắng Dưới Chân Mây     title        0.4      0.222       0.8     0.444
              nếu ngày mai em rời xa anh     lyric        0.0      0.000       0.0     0.000
 họ gọi thành phố này là nơi khó tồn tại     lyric        0.0      0.000       0.4     1.000
vầng trăng treo trên núi nghiêng nghiêng     lyric        0.6      1.000       0.6     1.000
                                lac troi no_accent        0.0      0.000       0.0     0.000
       

Lưu ý: R@5 bị chặn trên vì nhiều bài có > 5 phiên bản (vd. Lạc Trôi có 8) → R@5 tối đa = 5/8. P@5 cũng bị chặn khi bài chỉ có 4 phiên bản (P@5 tối đa = 0.8).

## Step 7 – So sánh các chiến lược tiền xử lý

In [ ]:
STRATS = ['baseline', 'no_stopwords', 'stemmed', 'lemmatized', 'no_diacritics', 'no_stopwords+no_diacritics']
base_tokens = sum(len(d) for d in docs_base)
rows, engines = [], {}
for st in STRATS:
    prep = lambda t, st=st: tokenize_for_search(t, st)
    docs = [tokenize_for_search(t, st, already_clean=True) for t in corpus['search_text']]
    eng = SearchEngine(docs); engines[st] = (eng, prep)
    n_tok = sum(len(d) for d in docs)
    r = {'strategy': st, 'vocab_size': len(set(w for d in docs for w in d)), 'total_tokens': n_tok,
         'token_reduction_%': round(100 * (1 - n_tok / base_tokens), 2)}
    for m in ['tfidf', 'bm25']:
        e = evaluate(eng, prep, m)
        r[f'P@5_{m}'], r[f'R@5_{m}'] = round(e['P@5'].mean(), 3), round(e['R@5'].mean(), 3)
    rows.append(r)
res7 = pd.DataFrame(rows)
print(res7.to_string(index=False))

                  strategy  vocab_size  total_tokens  token_reduction_%  P@5_tfidf  R@5_tfidf  P@5_bm25  R@5_bm25
                  baseline        9712        836057               0.00       0.46      0.377      0.58     0.524
              no_stopwords        9512        620539              25.78       0.48      0.388      0.58     0.524
                   stemmed        9512        620539              25.78       0.48      0.388      0.58     0.524
                lemmatized        9512        620539              25.78       0.48      0.388      0.58     0.524
             no_diacritics        6664        836057               0.00       0.50      0.403      0.64     0.564
no_stopwords+no_diacritics        6566        620539              25.78       0.52      0.414      0.64     0.564


In [ ]:
# Giu co dinh method = BM25, doi preprocessing: top-5 id cho vai query
for q in ['lac troi', 'em la ai tu dau buoc den noi day', 'Thiệp Hồng Sai Tên']:
    print(f"\n'{q}'  (relevant: {QRELS.get(q)})")
    for st in STRATS:
        eng, prep = engines[st]
        print(f"  {st:<27} {[i for i,_ in eng.search(prep(q), 'bm25')]}")


'lac troi'  (relevant: [534, 536, 1128, 1145, 1278, 1608, 2708, 2729])
  baseline                    []
  no_stopwords                []
  stemmed                     []
  lemmatized                  []
  no_diacritics               [536, 1128, 1145, 1278, 1608]
  no_stopwords+no_diacritics  [536, 1128, 1145, 1278, 1608]

'em la ai tu dau buoc den noi day'  (relevant: [42, 683, 1129, 1146, 2741])
  baseline                    [1664, 323, 500, 2097, 1488]
  no_stopwords                [323, 1664, 500, 1488, 2097]
  stemmed                     [323, 1664, 500, 1488, 2097]
  lemmatized                  [323, 1664, 500, 1488, 2097]
  no_diacritics               [1524, 2548, 245, 34, 240]
  no_stopwords+no_diacritics  [658, 889, 2263, 2282, 559]

'Thiệp Hồng Sai Tên'  (relevant: [11, 75, 117, 118, 120, 122, 123, 124])
  baseline                    [118, 122, 117, 123, 120]
  no_stopwords                [118, 122, 117, 123, 120]
  stemmed                     [118, 122, 117, 123, 120]
  lemm

In [ ]:
# Luu ket qua
per_query = []
for st in STRATS:
    eng, prep = engines[st]
    for m in ['tfidf', 'bm25']:
        e = evaluate(eng, prep, m); e.insert(0, 'method', m); e.insert(0, 'strategy', st); per_query.append(e)
pd.concat(per_query).to_csv('Quy_ket_qua.csv', index=False, encoding='utf-8-sig')
res7.to_csv('Quy_tong_hop.csv', index=False, encoding='utf-8-sig')
print("Da luu Quy_ket_qua.csv (theo tung query) va Quy_tong_hop.csv")

Da luu Quy_ket_qua.csv (theo tung query) va Quy_tong_hop.csv


## Step 8 – Bảng kết quả, kết luận, hạn chế

In [ ]:
b = res7.set_index('strategy')
table = pd.DataFrame({
    'Strategy': STRATS,
    'Vocabulary size': b['vocab_size'].values,
    'Token reduction (%)': b['token_reduction_%'].values,
    'TF-IDF P@5': b['P@5_tfidf'].values, 'TF-IDF R@5': b['R@5_tfidf'].values,
    'BM25 P@5': b['P@5_bm25'].values, 'BM25 R@5': b['R@5_bm25'].values})
try:
    print(table.to_markdown(index=False))
except ImportError:   # chua cai tabulate
    print(table.to_string(index=False))
best = table.loc[table['BM25 P@5'].idxmax(), 'Strategy']
print("\nChien luoc tot nhat theo BM25 P@5:", best)

| Strategy                   |   Vocabulary size |   Token reduction (%) |   TF-IDF P@5 |   TF-IDF R@5 |   BM25 P@5 |   BM25 R@5 |
|:---------------------------|------------------:|----------------------:|-------------:|-------------:|-----------:|-----------:|
| baseline                   |              9712 |                  0    |         0.46 |        0.377 |       0.58 |      0.524 |
| no_stopwords               |              9512 |                 25.78 |         0.48 |        0.388 |       0.58 |      0.524 |
| stemmed                    |              9512 |                 25.78 |         0.48 |        0.388 |       0.58 |      0.524 |
| lemmatized                 |              9512 |                 25.78 |         0.48 |        0.388 |       0.58 |      0.524 |
| no_diacritics              |              6664 |                  0    |         0.5  |        0.403 |       0.64 |      0.564 |
| no_stopwords+no_diacritics |              6566 |                 25.78 |         

### Kết luận (số dưới đây là lần chạy hiện tại – cập nhật lại nếu chạy lại ra khác)
- **TF-IDF vs BM25 (cố định baseline):** BM25 tốt hơn: P@5 0.58 vs 0.46, R@5 0.524 vs 0.377. Khác biệt rõ ở "Lạc Trôi" (1.0 vs 0.6), "Nắng Dưới Chân Mây" (0.8 vs 0.4), "họ gọi thành phố này…" (0.4 vs 0). BM25 bão hoà tf và chuẩn hoá theo avgdl nên bài lời dài lặp nhiều từ phổ biến ("em", "anh") không chiếm top.
- **Theo loại truy vấn (BM25):** tên bài gần như hoàn hảo (P@5 ≈ 0.96); câu hát yếu hơn (0.33) vì câu hát dùng từ phổ biến; gõ không dấu = 0 với index có dấu.
- **Preprocessing (cố định BM25):** `no_diacritics` cho P@5 cao nhất (0.64): cứu được "lac troi" (0 → 5/5), giảm vocabulary ~31% (9712 → 6664) vì gộp các âm tiết khác dấu. Nhưng "em la ai tu dau buoc den noi day" vẫn 0 vì sau khi bỏ dấu các từ quá phổ biến/nhập nhằng (la = là/lá/lạ, day = đây/dây/day).
- Bỏ stop word giảm 25.8% token nhưng P@5 BM25 không đổi (0.58) → tiết kiệm index mà không mất chất lượng trên bộ truy vấn này.
- `stemmed`/`lemmatized` gần như không đổi gì so với `no_stopwords` vì kho chủ yếu là tiếng Việt: Porter chỉ cắt hậu tố tiếng Anh (`stem("ngày")` = `ngày`), WordNet không có từ tiếng Việt.

### Hạn chế
- Bộ nhãn nhỏ (~10 truy vấn), chỉ nhãn nhị phân "đúng bài/không"; chưa phân biệt bản gốc / remix / live / beat (nên dùng nhãn mức độ 0–1–2 và nDCG).
- Tiếng Việt bị tách theo âm tiết ("tình yêu" → tình, yêu) → mất cụm từ; nên thử bigram hoặc tách từ (underthesea/pyvi).
- ~23% bài không có lời → chỉ khớp được qua title/artist.
- Dữ liệu trùng (cùng 1 bài nhiều link) làm top-5 bị lấp bởi bản trùng — tốt cho P@5 nhưng xấu cho trải nghiệm người dùng.

### Thảo luận
- **Vì sao 2 chiến lược cho top khác nhau?** Mỗi chiến lược tạo vocabulary khác: bỏ stop word làm đổi độ dài bài (đổi avgdl, chuẩn hoá) và mất token khớp; bỏ dấu gộp nhiều từ khác nghĩa thành một term → idf đổi, thứ hạng đổi.
- **BM25 xử lý độ dài bài:** điểm mỗi term = idf · f·(k1+1) / (f + k1·(1 − b + b·|d|/avgdl)). k1 = 1.5 làm tf bão hoà (lặp điệp khúc 20 lần không gấp 20 điểm); b = 0.75 phạt bài dài hơn avgdl, b = 0 thì không phạt.
- **Vì sao không so điểm BM25 với cosine trực tiếp?** Cosine nằm trong [0, 1], BM25 không có chặn trên và phụ thuộc độ dài query, idf của kho → thang đo khác nhau; chỉ so được *thứ hạng* qua P@k, R@k (hoặc chuẩn hoá điểm trước khi trộn).